# Retrieval-Augmented Generation (RAG) Pipeline

## 1. Installation

In [1]:
%pip install -q -U ragas bitsandbytes transformers accelerate langchain chromadb \
    langchain-huggingface datasets python-dotenv langchain-text-splitters langchain-community \
    sentence-transformers

## 2. Imports

In [2]:
import os
import torch
from datasets import load_dataset, Dataset
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import pipeline as hf_pipeline
from sentence_transformers import CrossEncoder
from dotenv import load_dotenv
import pandas as pd

## 3. Data Loading & Preprocessing
We use the IBM WatsonX documentation dataset.

In [3]:
# Load the IBM corpus dataset
corpus_dataset = load_dataset("ibm-research/watsonxDocsQA", "corpus", split="train")
print(type(corpus_dataset[0]))

# Convert into LangChain Document objects
documents = []
source_dict = {}
for entry in corpus_dataset:
    content = f"Title: {entry['title']}\n\n{entry['document']}"
    doc = Document(page_content=content, metadata={"source": entry['doc_id']})
    source_dict[entry['doc_id']] = content
    documents.append(doc)

print(f"Loaded {len(documents)} documents.")

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=300)
docs = text_splitter.split_documents(documents)
print(f"Split into {len(docs)} chunks.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


<class 'dict'>
Loaded 1144 documents.
Split into 7212 chunks.


## 4. Vector Database Construction
Using sentence-transformers to create embeddings and ChromaDB for storage.

In [4]:
# Initialize embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

# Process in batches to manage memory
batch_size = 100
vectorstore = None

print(f"Starting embedding for {len(docs)} chunks...")

for i in range(0, len(docs), batch_size):
    batch = docs[i : i + batch_size]
    if vectorstore is None:
        vectorstore = Chroma.from_documents(documents=batch, embedding=embeddings)
    else:
        vectorstore.add_documents(batch)
    print(f"✅ Completed {i + len(batch)} / {len(docs)} chunks...")

print("✅ Vector store ready.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Starting embedding for 7212 chunks...
✅ Completed 100 / 7212 chunks...
✅ Completed 200 / 7212 chunks...
✅ Completed 300 / 7212 chunks...
✅ Completed 400 / 7212 chunks...
✅ Completed 500 / 7212 chunks...
✅ Completed 600 / 7212 chunks...
✅ Completed 700 / 7212 chunks...
✅ Completed 800 / 7212 chunks...
✅ Completed 900 / 7212 chunks...
✅ Completed 1000 / 7212 chunks...
✅ Completed 1100 / 7212 chunks...
✅ Completed 1200 / 7212 chunks...
✅ Completed 1300 / 7212 chunks...
✅ Completed 1400 / 7212 chunks...
✅ Completed 1500 / 7212 chunks...
✅ Completed 1600 / 7212 chunks...
✅ Completed 1700 / 7212 chunks...
✅ Completed 1800 / 7212 chunks...
✅ Completed 1900 / 7212 chunks...
✅ Completed 2000 / 7212 chunks...
✅ Completed 2100 / 7212 chunks...
✅ Completed 2200 / 7212 chunks...
✅ Completed 2300 / 7212 chunks...
✅ Completed 2400 / 7212 chunks...
✅ Completed 2500 / 7212 chunks...
✅ Completed 2600 / 7212 chunks...
✅ Completed 2700 / 7212 chunks...
✅ Completed 2800 / 7212 chunks...
✅ Completed 2900 / 

## 5. Load Mistral-7B-Instruct (4-bit quantized)

In [5]:
# Load HuggingFace token from .env
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)

print(f"✅ Model loaded on device: {model.device}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Model loaded on device: cuda:0


## 6. Retriever & Reranker Setup

In [6]:
# MMR retriever — fetches a diverse pool, reranker picks the best
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 15,          # final candidates passed to reranker
        "fetch_k": 50,    # initial pool fetched via similarity search
        "lambda_mult": 0.6  # 1.0 = pure similarity, 0.0 = max diversity
    }
)

# Cross-encoder reranker — more accurate than bi-encoder for final scoring
rerank_model = CrossEncoder('BAAI/bge-reranker-base', device="cpu")

print("✅ Retriever and reranker ready.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Retriever and reranker ready.


## 7. RAG Pipeline

In [7]:
def rag_pipeline(question):
    initial_docs = retriever.invoke(question)
    if not initial_docs:
        return "No relevant documents found.", []

    pairs = [[question, doc.page_content] for doc in initial_docs]
    scores = rerank_model.predict(pairs)
    scored_docs = sorted(zip(scores, initial_docs), key=lambda x: x[0], reverse=True)
    top_docs = [doc for _, doc in scored_docs[:3]]
    print(top_docs,"top 3 docs")
    context = "\n\n".join([doc.page_content for doc in top_docs])
    # messages = [
    #     {
    #         "role": "user",
    #         "content": f"Context: {context}\n\nQuestion: {question}\n\nAnswer using only the provided context."
    #     }
    # ]
    # messages = [
    #         {
    #             "role": "user",
    #             "content": f"""
    #     You are a retrieval-based assistant.

    #     Use ONLY the provided context to answer the question.

    #     Rules:
    #     - Do NOT use prior knowledge.
    #     - Do NOT infer beyond explicit statements.
    #     - Do not infer associations unless explicitly stated.
    #     - If the answer is not clearly available in the context, say:
    #     "The provided context does not contain enough information to answer this question."
    #     - Be concise and factual.

    #     Context:
    #     {context}

    #     Question:
    #     {question}
    #     """
    #         }
    #     ]
    messages = [
        {
        "role": "system",
        "content": "You are a strict RAG system. You ONLY use explicitly stated context. No external knowledge."
        },
        {
        "role": "user",
        "content": f"""
        Step 1: Extract exact relevant lines from context.
        Step 2: If none exist, say "Insufficient context."
        Step 3: Answer ONLY using extracted lines, be concise and factual.
        Step 4: Do not combine multiple components into a higher-level product definition unless explicitly stated in context.

        Context:
        {context}

        Question:
        {question}
        """
        }
    ]
    

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True        
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,                            
            max_new_tokens=400,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    
    prompt_length = inputs["input_ids"].shape[-1]
    answer = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

    return answer, top_docs

## 8. Quick Test

In [8]:
question = "Tell me about India?"
print(f"Question: {question}")
print("-" * 30)
answer, used_docs = rag_pipeline(question)
print(f"Answer: {answer}")
print(f"\n({len(used_docs)} chunks used by the LLM)")

Question: Tell me about India?
------------------------------


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[Document(metadata={'source': '174D6FDF73627D7B2258D7F351C3D0156C06D1DC'}, page_content='Television                Factual TV                          \n Television                Holiday TV                          \n Television                Music TV                            \n Television                Reality TV                          \n Television                Science Fiction TV                  \n Travel                   \n Travel                    Travel Accessories                  \n Travel                    Travel Locations                    \n Travel                    Travel Locations                     Africa Travel                                  \n Travel                    Travel Locations                     Asia Travel                                    \n Travel                    Travel Locations                     Australia and Oceania Travel                   \n Travel                    Travel Locations                     Europe Travel'), Document(

## 9. Load Test Dataset

In [9]:
# FIX: corrected typo from 'datset_test' → 'dataset_test'
# Use the test split for proper evaluation
from datasets import load_dataset

dataset_test = load_dataset(
    "ibm-research/watsonxDocsQA",
    "question_answers",
    split="test"
)
print(dataset_test[0])
print(f"Total test samples: {len(dataset_test)}")


{'question_id': 'test_1', 'question': 'What foundation models are available in watsonx.ai ?', 'correct_answer': 'The following models are available in watsonx.ai: \nflan-t5-xl-3b\nFlan-t5-xxl-11b\nflan-ul2-20b\ngpt-neox-20b\ngranite-13b-chat-v2\ngranite-13b-chat-v1\ngranite-13b-instruct-v2\ngranite-13b-instruct-v1\nllama-2-13b-chat\nllama-2-70b-chat\nmpt-7b-instruct2\nmt0-xxl-13b\nstarcoder-15.5b', 'correct_answer_document_ids': '5B37710FE7BBD6EFB842FEB7B49B036302E18F81', 'ground_truths_contexts': 'Supported foundation models available with watsonx.ai \n\nA collection of open source and IBM foundation models are deployed in IBM watsonx.ai.\n\nThe following models are available in watsonx.ai:\n\n\n\n*  flan-t5-xl-3b\n*  flan-t5-xxl-11b\n*  flan-ul2-20b\n*  gpt-neox-20b\n*  granite-13b-chat-v2\n*  granite-13b-chat-v1\n*  granite-13b-instruct-v2\n*  granite-13b-instruct-v1\n*  llama-2-13b-chat\n*  llama-2-70b-chat\n*  mpt-7b-instruct2\n*  mt0-xxl-13b\n*  starcoder-15.5b\n\n\n\nYou can pro

## 10. Retrieval Accuracy: Hit@1 and Hit@3 with Reranker

In [11]:
def check_retrieval(question, true_id, k=3):
    docs = retriever.invoke(question)

    if not docs:
        print(f"No results for: {question[:40]}")
        return False, False

    pairs = [[question, d.page_content] for d in docs]
    scores = rerank_model.predict(pairs)

    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    top_docs = [d for _, d in ranked[:k]]

    hit_at_1 = False
    hit_at_3 = False

    print("\n------------------------------")
    print("Q:", question)
    print("Expected:", true_id)
    print("------------------------------")

    true_id = str(true_id).strip()

    for idx, doc in enumerate(top_docs):
        pred_id = str(doc.metadata.get("source")).strip()

        match = pred_id == true_id

        if match:
            if idx == 0:
                hit_at_1 = True
            hit_at_3 = True

        print(f"{idx+1}. {pred_id} | match={match}")

    return hit_at_1, hit_at_3


hits1, hits3 = [], []

print("running eval...\n")

for i, row in enumerate(dataset_test.select(range(30))):
    q = row["question"]
    target = row["correct_answer_document_ids"]

    h1, h3 = check_retrieval(q, target, k=3)

    hits1.append(h1)
    hits3.append(h3)

    if (i + 1) % 10 == 0:
        print(f"processed {i+1} samples...")


h1_score = sum(hits1) / len(hits1)
h3_score = sum(hits3) / len(hits3)

print("\n====================")
print("FINAL RESULTS")
print("====================")
print("Hit@1:", round(h1_score, 3))
print("Hit@3:", round(h3_score, 3))

if h1_score > 0.85:
    print("\nlooks good — top result seems reliable enough")
    print("you can probably pass full doc to the LLM")

running eval...


------------------------------
Q: What foundation models are available in watsonx.ai ?
Expected: 5B37710FE7BBD6EFB842FEB7B49B036302E18F81
------------------------------
1. 5B37710FE7BBD6EFB842FEB7B49B036302E18F81 | match=True
2. B2593108FA446C4B4B0EF5ADC2CD5D9585B0B63C | match=False
3. 5B37710FE7BBD6EFB842FEB7B49B036302E18F81 | match=True

------------------------------
Q: What is greedy decoding?
Expected: 42AE491240EF740E6A8C5CF32B817E606F554E49
------------------------------
1. 42AE491240EF740E6A8C5CF32B817E606F554E49 | match=True
2. 43785386700CF73E37A8F76ADC4EF9FB01EE0AEB | match=False
3. E5D702E67E93752155510B56A3B2F464E190EBA2 | match=False

------------------------------
Q: What tuning parameters are available for IBM foundation models?
Expected: 51747F17F413F1F34CFD73D170DE392D874D03DD
------------------------------
1. 51747F17F413F1F34CFD73D170DE392D874D03DD | match=True
2. C32FE380CF3083B6D85554063B5ACB153FC1C8BE | match=False
3. 5BC1631D896899D03E7D8DD2296

## 11. Build RAGAS Evaluation Dataset

We now call `rag_pipeline()` once per question and unpack both
the answer and the exact chunks used — no more double retrieval.

In [ ]:
questions = []
answers = []
contexts = []
ground_truths = []

print("Generating responses for RAGAS evaluation...")

for row in dataset_test.select(range(20)):
    question = row["question"]

    # FIX: single call — answer + the exact reranked docs the LLM used
    response, top_docs = rag_pipeline(question)

    questions.append(question)
    answers.append(response)
    contexts.append([doc.page_content for doc in top_docs])  # actual docs used
    ground_truths.append(row["correct_answer"])

    print(f"✅ Done: {question[:60]}...")

# FIX: use 'reference' key (required by RAGAS >= 0.1.9)
# If you're on an older version and get a KeyError, change back to 'ground_truth'
data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "reference": ground_truths,
}

eval_dataset = Dataset.from_dict(data)
print(f"\n✅ eval_dataset created with {len(eval_dataset)} rows.")
for i in range(len(eval_dataset)):
    print(eval_dataset[i])